# 04 — Comparaison des distances pour recommander


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd().resolve()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "ml-research"
df = pd.read_csv(DATA / "plants_SYNTHETIC_DEMO.csv")
print("Synthetic demo — NOT horticultural evidence:", df.shape)
df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
NUM = ["height_cm", "spread_cm", "hardiness_min_c"]
CAT = ["plant_type", "evergreen", "sun_exposure", "soil_type"]
pre = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("category", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), CAT)
])
X = pre.fit_transform(df[NUM + CAT])
print(X.shape)

In [ ]:
from sklearn.neighbors import NearestNeighbors
# Demonstration query: choose an existing synthetic plant as a proxy.
# In production, encode a garden preference vector in the SAME feature space.
query = X[0:1]
results = {}
for metric in ["euclidean","manhattan","cosine"]:
    nn = NearestNeighbors(n_neighbors=6,metric=metric).fit(X)
    dist, indices = nn.kneighbors(query)
    results[metric] = list(zip(df.iloc[indices[0][1:]]["scientific_name"].tolist(),
                               np.round(dist[0][1:],3).tolist()))
for metric, ranked in results.items():
    print(metric, ranked)
print("No winner without independently assessed garden-plant relevance labels.")

In [ ]:
# With relevance labels, evaluate Precision@3 / NDCG@3 by garden scenario.
# Do not confuse closest botanical neighbors with suitability for a garden.